# 06 Evaluation: costly errors with RGB and 13 bands

**Purpose.** Compare all three models (random forest, CNN, ResNet18) on RGB and 13 bands with the same test patches, and answer the sub-questions: does the 13-band advantage hold for every model (SQ1), which confusions are costly for a crop insurer and how often each model makes them (SQ2), and do the 13 bands reduce them (SQ3). Nothing is trained here: the notebook reads the saved test predictions of notebooks 03 to 05.

**Functions.** `src/piece_of_crop/evaluate.py` (`insurer_cost_matrix`, `listed_critical_confusions`, `load_predictions`, `block_bootstrap_ci`) and `data.py` (`read_split`, `get_cache`, `to_rgb_image`).

**Reads.** `results/split.csv`, `results/insurer_cost_matrix.csv`, the 18 prediction files in `results/predictions/` (3 models x 2 inputs x 3 seeds) and the patch cache `data/eurosat_cache.npy` for the images in section 8.

**Saves.** `results/evaluation_runs.csv`, `evaluation_per_class.csv`, `evaluation_differences.csv`, `evaluation_confused_pairs.csv`, `evaluation_sensitivity.csv`, `evaluation_review.csv` and the figures `results/figures/evaluation_*.png`.

## 1. Setup and the saved predictions

Every prediction file must contain the 3,854 test patches of `split.csv` in the same order and with the same labels. This makes the comparison paired: RGB and 13 bands, and the three models, are judged on exactly the same patches, so a difference between them cannot come from easier or harder test patches.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from IPython.display import display

from piece_of_crop import BAND_NAMES, CLASS_NAMES, MODEL_FILES, read_split, get_cache, to_rgb_image
from piece_of_crop import insurer_cost_matrix, listed_critical_confusions, load_predictions, block_bootstrap_ci

SEEDS = (0, 1, 2)
INPUTS = ("rgb", "13band")
MODELS = tuple(MODEL_FILES)  # RandomForest, SmallCNN, ResNet18
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 0

project_root = Path.cwd().parents[0]
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"

split = read_split(results_dir / "split.csv")
test_rows = split[split["split"].eq("test")].reset_index(drop=True)
predictions = load_predictions(predictions_dir, test_rows, SEEDS)
y_true = test_rows["label"].to_numpy()
blocks = test_rows["block"].to_numpy()
label_ids = np.arange(len(CLASS_NAMES))

print("Prediction files:", len(predictions))
print("Test patches:", len(test_rows), "in", test_rows["block"].nunique(), "spatial blocks")

## 2. The insurer cost matrix

A crop insurer checks whether a parcel is the land use its owner declared before pricing a policy or paying a claim. Not every mistake of a model costs the same in that check, so we weigh each confusion with a cost, in severity points: rows are the true class, columns the predicted class, and the expected cost of a model is the average over its predictions. This is the loss matrix of decision theory, where "the consequences of these two types of mistake can be dramatically different" (Bishop, *Pattern Recognition and Machine Learning*, section 1.5.2, page 41).

| Cost | Confusion | Why it matters to an insurer |
|---|---|---|
| 9 | agricultural land (AnnualCrop, PermanentCrop, Pasture, HerbaceousVegetation) with Forest or built-up land (Highway, Industrial, Residential) | the basic use of the parcel is wrong: the insurer would cover land that is not farmed, or turn away farmland |
| 8 | a crop class (AnnualCrop, PermanentCrop) with another agricultural class | the land is farmed, but the insured crop is wrong, and orchards, cereals and grazing land carry different risks and premiums |
| 6 | Pasture with HerbaceousVegetation | both are grassland, so the risk differs less, but managed grazing land is insured differently from natural grass |
| 2 | every other mistake (for example River with Highway) | wrong, but outside agricultural insurance |

We call the confusions with a cost of 8 or 9 **critical**: they involve farmland and change either whether the parcel is farmed or what grows on it.

**When the weights were set.** Our plan was to write the costs down before training. The matrix was first committed on 22 November 2026, together with results from an earlier split that we later replaced. It was therefore fixed before all results in this notebook, which come from the restored split (notebooks 03 to 05, 24 to 26 November), and we do not change any weight now. The cell below checks that the matrix in `src/` is still the committed table.

**Counterargument.** The points are not euros and do not come from insurance data; another insurer could weigh the confusions differently. We accept this. Section 7 tests whether the conclusions change with other weights, including the earlier list of critical confusions, which left out HerbaceousVegetation confused with Forest or built-up land.

In [ ]:
costs = insurer_cost_matrix()
critical = costs >= 8
committed = pd.read_csv(results_dir / "insurer_cost_matrix.csv", index_col=0)
assert np.array_equal(committed.to_numpy(), costs), "the cost matrix differs from the committed table"

display(pd.DataFrame(costs, index=CLASS_NAMES, columns=CLASS_NAMES).astype(int))
print("Critical confusions (cost 8 or 9):", int(critical.sum()), "of 90 possible mistakes")

## 3. All models and both inputs on the test set

For every run: accuracy, macro-F1, the share of test patches with a critical error, and the average cost per test patch. Macro-F1 averages the F1 score of the 10 classes with equal weight, so a weak class counts as much as a strong one (lecture_2_introduction_to_ML.pdf, slide 34). The table shows the mean and standard deviation over the three seeds.

In [ ]:
run_rows, class_rows = [], []
for (model, input_type, seed), table in predictions.items():
    y_pred = table["y_pred"].to_numpy()
    run_rows.append({"model": model, "input": input_type, "seed": seed,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "critical_error_rate": critical[y_true, y_pred].mean(),
        "cost_per_patch": costs[y_true, y_pred].mean()})
    for class_id, f1 in zip(label_ids, f1_score(y_true, y_pred, average=None, labels=label_ids)):
        class_rows.append({"model": model, "input": input_type, "seed": seed, "class": CLASS_NAMES[class_id], "f1": f1})

runs = pd.DataFrame(run_rows)
per_class = pd.DataFrame(class_rows)
runs.to_csv(results_dir / "evaluation_runs.csv", index=False)
per_class.to_csv(results_dir / "evaluation_per_class.csv", index=False)

comparison = runs.groupby(["model", "input"], sort=False)[["accuracy", "macro_f1", "critical_error_rate", "cost_per_patch"]]
display(comparison.agg(["mean", "std"]).round(4))

print("F1 per class, mean over the seeds:")
display(per_class.groupby(["class", "model", "input"], sort=False)["f1"].mean().unstack(["model", "input"]).round(3))

In [ ]:
# main comparison figure: macro-F1 and cost per patch
summary = runs.groupby(["model", "input"], sort=False)[["macro_f1", "cost_per_patch"]].agg(["mean", "std"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(MODELS))
for ax, metric, label in ((axes[0], "macro_f1", "Macro-F1"), (axes[1], "cost_per_patch", "Cost per test patch (points)")):
    for k, input_type in enumerate(INPUTS):
        means = [summary.loc[(m, input_type), (metric, "mean")] for m in MODELS]
        stds = [summary.loc[(m, input_type), (metric, "std")] for m in MODELS]
        ax.bar(x + (k - 0.5) * 0.38, means, width=0.38, yerr=stds, capsize=3, label=input_type)
    ax.set_xticks(x, MODELS)
    ax.set(title=f"{label} on the test set (mean of 3 seeds)", xlabel="Model", ylabel=label)
    ax.grid(alpha=0.3, axis="y")
    ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / "evaluation_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Confusion matrices

One matrix per model and input, summed over the three seeds and divided by the number of patches of each true class, so each row shows where the patches of that class went (lecture_2_introduction_to_ML.pdf, slide 35). The diagonal is the recall of each class. Cells with a critical confusion are framed in red.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 19))
short = [name[:10] for name in CLASS_NAMES]
for row, model in enumerate(MODELS):
    for col, input_type in enumerate(INPUTS):
        counts = sum(confusion_matrix(y_true, predictions[(model, input_type, s)]["y_pred"], labels=label_ids) for s in SEEDS)
        rates = counts / counts.sum(axis=1, keepdims=True)
        ax = axes[row, col]
        ax.imshow(rates, cmap="Blues", vmin=0, vmax=1)
        for i in label_ids:
            for j in label_ids:
                if rates[i, j] >= 0.005:
                    ax.text(j, i, f"{rates[i, j]:.2f}", ha="center", va="center", fontsize=7,
                            color="white" if rates[i, j] > 0.5 else "black")
                if critical[i, j]:
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="red", lw=0.8))
        ax.set_xticks(label_ids, short, rotation=60, fontsize=8)
        ax.set_yticks(label_ids, short, fontsize=8)
        ax.set(title=f"{model}, {input_type}", xlabel="Predicted class", ylabel="True class")
fig.suptitle("Test confusion matrices, share of each true class (3 seeds summed; red = critical)", y=1.0)
fig.tight_layout()
fig.savefig(figures_dir / "evaluation_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 13 bands minus RGB, with intervals

For every model and test patch we take the 13-band result minus the RGB result, averaged over the three seeds, for three measures: correct or not (accuracy), critical error or not, and cost. A positive accuracy difference and negative critical-error and cost differences favour the 13 bands.

**Why a block bootstrap.** The three seeds show how much training varies, but not how much the result depends on which test patches we happen to have. A bootstrap estimates this by drawing the test set again many times with replacement and recomputing the mean (external: Efron and Tibshirani, 1993, *An Introduction to the Bootstrap*, chapter 6). Neighbouring patches in one 0.5° block share fields, weather and season, so they are not independent; we therefore draw whole blocks, as for clustered data (external: Field and Welsh, 2007, "Bootstrapping clustered data", *Journal of the Royal Statistical Society B*, 69(3)). The 2.5th and 97.5th percentiles of 2,000 redrawn means give a 95% interval. If it does not contain 0, the difference is unlikely to be an accident of this test set.

In [ ]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
measures = {"accuracy": lambda p: (p == y_true).astype(float),
            "critical_error_rate": lambda p: critical[y_true, p].astype(float),
            "cost_per_patch": lambda p: costs[y_true, p]}

difference_rows = []
for model in MODELS:
    for measure, per_patch in measures.items():
        per_seed = np.array([per_patch(predictions[(model, "13band", s)]["y_pred"].to_numpy())
                             - per_patch(predictions[(model, "rgb", s)]["y_pred"].to_numpy()) for s in SEEDS])
        low, high = block_bootstrap_ci(per_seed.mean(axis=0), blocks, N_BOOTSTRAP, rng)
        difference_rows.append({"model": model, "measure": measure, "13band_minus_rgb": per_seed.mean(),
                                "seed_std": per_seed.mean(axis=1).std(ddof=1), "ci_low": low, "ci_high": high,
                                "excludes_0": not (low <= 0 <= high)})

differences = pd.DataFrame(difference_rows)
differences.to_csv(results_dir / "evaluation_differences.csv", index=False)
display(differences.round(4))

## 6. Which confusions happen, and how often

The directed confusions (true class predicted as another class), as the number of test patches per seed, for every model and input. The first table ranks the most frequent mistakes of each model and input; the second shows every critical confusion side by side, which is where the 13 bands either help an insurer or not.

In [ ]:
pair_rows = []
for (model, input_type, seed), table in predictions.items():
    counts = confusion_matrix(y_true, table["y_pred"], labels=label_ids)
    for i in label_ids:
        for j in label_ids:
            if i != j and counts[i, j] > 0:
                pair_rows.append({"model": model, "input": input_type, "seed": seed, "true": CLASS_NAMES[i],
                                  "predicted": CLASS_NAMES[j], "patches": counts[i, j],
                                  "cost": costs[i, j], "critical": critical[i, j]})
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(results_dir / "evaluation_confused_pairs.csv", index=False)

per_seed = (pairs.groupby(["model", "input", "true", "predicted", "cost", "critical"], sort=False)["patches"].sum() / len(SEEDS)).reset_index()
print("Five most frequent confusions per model and input (patches per seed):")
display(per_seed.sort_values("patches", ascending=False).groupby(["model", "input"], sort=False).head(5)
        .sort_values(["model", "input"], kind="stable").round(1))

critical_table = (per_seed[per_seed["critical"]]
                  .pivot_table(index=["true", "predicted", "cost"], columns=["model", "input"], values="patches", fill_value=0)
                  .reindex(columns=pd.MultiIndex.from_product([MODELS, INPUTS])))
critical_table = critical_table[critical_table.max(axis=1) >= 1]
critical_table.loc[("Total", "", "")] = critical_table.sum()
print("Critical confusions, patches per seed (only those with at least 1 patch per seed for some model):")
display(critical_table.round(1))

## 7. Do the conclusions depend on the cost weights?

The same 13-band minus RGB difference in cost, with intervals, under five assumptions: every mistake costs the same (this is the error rate), the committed weights, critical confusions half as costly, critical confusions twice as costly, and the committed weights with only the earlier list of critical confusions doubled. If a conclusion keeps its sign under all five, it does not depend on our choice of weights.

In [ ]:
equal = 1.0 - np.eye(len(CLASS_NAMES))
listed = listed_critical_confusions()
scenarios = {"every mistake costs 1": equal,
             "committed weights": costs,
             "critical x 0.5": np.where(critical, costs * 0.5, costs),
             "critical x 2": np.where(critical, costs * 2, costs),
             "earlier critical list x 2": np.where(listed, costs * 2, costs)}

rng = np.random.default_rng(BOOTSTRAP_SEED)
sensitivity_rows = []
for scenario, matrix in scenarios.items():
    for model in MODELS:
        per_seed = np.array([matrix[y_true, predictions[(model, "13band", s)]["y_pred"]]
                             - matrix[y_true, predictions[(model, "rgb", s)]["y_pred"]] for s in SEEDS])
        low, high = block_bootstrap_ci(per_seed.mean(axis=0), blocks, N_BOOTSTRAP, rng)
        sensitivity_rows.append({"scenario": scenario, "model": model, "13band_minus_rgb": per_seed.mean(),
                                 "ci_low": low, "ci_high": high,
                                 "verdict": "13 bands cheaper" if high < 0 else ("RGB cheaper" if low > 0 else "no clear difference")})

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(results_dir / "evaluation_sensitivity.csv", index=False)
display(sensitivity.round(4))

## 8. Looking at the misclassified patches

Numbers alone do not show why a model fails, so we look at the patches themselves (true colour, bands 4, 3, 2, stretched for display only). Three groups, 30 patches in total:

1. **HerbaceousVegetation predicted as Forest by the 13-band ResNet but correct with RGB** (seed 0). Notebook 05 found that this mistake grows the more the ResNet relies on the extra bands.
2. **PermanentCrop predicted as HerbaceousVegetation by both ResNets** (seed 0), the most frequent mistake of the best model.
3. **The ten most costly mistakes of the 13-band CNN** (seed 0), the model where the 13 bands help most among the deep models.

Then a direct check of the hypothesis from notebook 05: if grassland is called forest because both reflect near-infrared strongly, the misclassified HerbaceousVegetation patches should have a Forest-like spectrum in the infrared bands.

In [ ]:
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
test_cache_rows = test_rows["cache_row"].to_numpy()
name_to_id = {name: k for k, name in enumerate(CLASS_NAMES)}


def show_patches(indices, title, predicted, n=10):
    """Show up to n test patches with their true and predicted class."""
    indices = list(indices)[:n]
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.6))
    for ax in axes:
        ax.axis("off")
    for ax, k in zip(axes, indices):
        ax.imshow(to_rgb_image(np.asarray(cache[test_cache_rows[k]], dtype=np.float32)))
        ax.set_title(f"{CLASS_NAMES[y_true[k]][:12]}\n-> {CLASS_NAMES[predicted[k]][:12]}", fontsize=8)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


resnet_rgb = predictions[("ResNet18", "rgb", 0)]["y_pred"].to_numpy()
resnet_13 = predictions[("ResNet18", "13band", 0)]["y_pred"].to_numpy()
cnn_13 = predictions[("SmallCNN", "13band", 0)]["y_pred"].to_numpy()
herb, forest, perm = name_to_id["HerbaceousVegetation"], name_to_id["Forest"], name_to_id["PermanentCrop"]

group1 = np.where((y_true == herb) & (resnet_13 == forest) & (resnet_rgb == herb))[0]
group2 = np.where((y_true == perm) & (resnet_13 == herb) & (resnet_rgb == herb))[0]
cnn_costs = costs[y_true, cnn_13]
group3 = np.argsort(-cnn_costs, kind="stable")[:10]
print("Group sizes (seed 0):", len(group1), len(group2), "and the ten costliest CNN mistakes")

for indices, title, predicted, name in ((group1, "HerbaceousVegetation -> Forest, ResNet 13 bands (RGB correct)", resnet_13, "herb_forest"),
                                         (group2, "PermanentCrop -> HerbaceousVegetation, both ResNets", resnet_13, "perm_herb"),
                                         (group3, "Ten most costly mistakes, CNN 13 bands", cnn_13, "cnn_costliest")):
    fig = show_patches(indices, title, predicted)
    fig.savefig(figures_dir / f"evaluation_errors_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# hypothesis check: spectrum of the misclassified grassland against correct grassland and forest
def mean_spectrum(indices):
    return np.mean([np.asarray(cache[test_cache_rows[k]], dtype=np.float32).mean(axis=(1, 2)) for k in indices], axis=0)

all_seeds_herb_forest = sorted({k for s in SEEDS
                                for k in np.where((y_true == herb) & (predictions[("ResNet18", "13band", s)]["y_pred"].to_numpy() == forest))[0]})
groups = {"HerbaceousVegetation, called Forest (13-band ResNet, any seed)": all_seeds_herb_forest,
          "HerbaceousVegetation, correct (13-band ResNet, seed 0)": np.where((y_true == herb) & (resnet_13 == herb))[0],
          "Forest, correct (13-band ResNet, seed 0)": np.where((y_true == forest) & (resnet_13 == forest))[0]}

order = [BAND_NAMES.index(b) for b in ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]]
fig, ax = plt.subplots(figsize=(10, 4))
spectra = {}
for label, indices in groups.items():
    spectra[label] = mean_spectrum(indices)
    ax.plot([BAND_NAMES[b] for b in order], spectra[label][order], marker="o", label=f"{label} (n={len(indices)})")
ax.set(title="Mean reflectance per band (bands in wavelength order)", xlabel="Band", ylabel="Mean raw value")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(figures_dir / "evaluation_herb_forest_spectra.png", dpi=150, bbox_inches="tight")
plt.show()
display(pd.DataFrame(spectra, index=BAND_NAMES).T[[BAND_NAMES[b] for b in order]].round(0))

## 9. Sending uncertain patches to a human

The system is meant to flag parcels for human review, not to decide claims. The course's "reject option" leaves the cases where the model is least sure to a human expert, which lowers the error rate on the cases it does decide (Bishop, section 1.5.3, page 42). We sort the test patches of each run by the model's confidence (its highest class probability), send the least confident share to review, and measure how many of the critical errors that catches. The curve shows this for review budgets from 0% to 30% of the patches, and the table for 5%, 10% and 20%.

In [ ]:
budgets = np.linspace(0, 0.30, 31)
review_rows = []
fig, ax = plt.subplots(figsize=(8, 4.5))
for model in MODELS:
    for input_type in INPUTS:
        caught = []
        for seed in SEEDS:
            table = predictions[(model, input_type, seed)]
            is_critical_error = critical[y_true, table["y_pred"].to_numpy()]
            order = np.argsort(table["confidence"].to_numpy(), kind="stable")  # least confident first
            caught_by_rank = np.cumsum(is_critical_error[order]) / max(1, is_critical_error.sum())
            caught.append([caught_by_rank[int(b * len(order)) - 1] if b > 0 else 0.0 for b in budgets])
        caught = np.mean(caught, axis=0)
        ax.plot(budgets * 100, caught * 100, ls="-" if input_type == "13band" else "--", label=f"{model} {input_type}")
        for budget in (0.05, 0.10, 0.20):
            review_rows.append({"model": model, "input": input_type, "review_share": budget,
                                "critical_errors_caught": caught[np.argmin(np.abs(budgets - budget))]})
ax.set(title="Critical errors caught by reviewing the least confident patches", xlabel="Share of test patches sent to review (%)",
       ylabel="Critical errors caught (%)", ylim=(0, 100))
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(figures_dir / "evaluation_review_curve.png", dpi=150, bbox_inches="tight")
plt.show()

review = pd.DataFrame(review_rows)
review.to_csv(results_dir / "evaluation_review.csv", index=False)
display(review.pivot_table(index=["model", "input"], columns="review_share", values="critical_errors_caught", sort=False).round(3))